# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [4]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [5]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
assert len(lines) == 4
assert lines['qty'].sum() == 7
lines


,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [6]:
# TODO
vendors = pd.json_normalize(vendors_json)

joined = pd.merge(lines, vendors, on='vendor_id', how='left', indicator=True)

unmatched = joined[joined['_merge'] == 'left_only']
unmatched_vendor = unmatched['vendor_id'].unique()[0]
unmatched_qty = unmatched['qty'].sum()

print(f"The unmatched vendor is: {unmatched_vendor}")
print(f"Units attached to this vendor: {unmatched_qty}")

joined

The unmatched vendor is: V-07
Units attached to this vendor: 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [7]:
# TODO
vendor_totals = joined.groupby('name', dropna=False)['qty'].sum().reset_index()
vendor_totals

,name,qty
0,Cav Merch,1
1,Hoos Burgers,3
2,NaN,3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [8]:
# TODO
zone_totals = joined.groupby('zone', dropna=False)['qty'].sum().reset_index()
print("Zone A sold the most items (4 units).")
print(f"The total across zones is {zone_totals['qty'].sum()}. The difference of 3 units went to the NaN zone (our unmatched vendor V-07).")

zone_totals

Zone A sold the most items (4 units).
The total across zones is 7. The difference of 3 units went to the NaN zone (our unmatched vendor V-07).


,zone,qty
0,A,4
1,NaN,3


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [10]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
clean_json = [order for order in ragged_json if 'lines' in order]

# Now flatten the cleaned list (no need for errors='ignore' here)
ragged_lines = pd.json_normalize(clean_json, record_path='lines', meta=['order', 'vendor_id'])

ragged_lines

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Fries,NaN,6,V-01


### Q6 — Validate

In [11]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

One row per line item is useful because the line item is the smallest thing that gets sold. Orders have different numbers of items, so one row per order would leave a list inside a cell, which is hard to do math on. With one row per line item, every row has one item and one qty, so I can sum, group, and filter normally. That made questions like "which zone sold the most items?" easy, and the total matched the expected 7.

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

A quantity of zero means the line was recorded and none were sold. A missing quantity means we don't know the number. If I used fillna(0), the Fries line would look like a confirmed zero. Totals and averages would then look complete when they aren't, and a report could say no Fries were sold when the data is just missing. Keeping NaN lets me flag those rows for follow-up and keeps the sums honest.


_your answer here_